# 01 · EDA
Point `CONFIG` at the active config. Findings go into `docs/approach.md` §2.

In [ ]:
import os, sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO); sys.path.insert(0, str(REPO))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils import load_config, read_split, get_path
CONFIG = ["configs/base.yaml"]  # add "configs/dryrun.yaml" for the 2025 dry run
cfg = load_config(CONFIG)
cols = cfg["columns"]
train, test = read_split(cfg, "train"), read_split(cfg, "test")
print(train.shape, test.shape)
train.head()

In [ ]:
pd.DataFrame({"train_null_%": train.isna().mean() * 100, "test_null_%": test.isna().mean() * 100}).round(2)

In [ ]:
# target distribution
y = train[cols["target"]]
print(y.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
if cfg["task"] == "regression":
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].hist(y.clip(upper=y.quantile(.99)), bins=100); ax[0].set_title("target (clipped at p99)")
    ax[1].hist(np.log1p(y.clip(lower=0)), bins=100); ax[1].set_title("log1p(target)")
    plt.show()
else:
    print(y.value_counts().head(30))

In [ ]:
# text length + a few raw examples
t = train[cols["text"]].fillna("").astype(str)
print(t.str.len().describe())
for s in t.sample(3, random_state=0):
    print("-" * 80); print(s[:800])

In [ ]:
# quantity parser coverage
from src.features_text import build_handcrafted
h = build_handcrafted(t.sample(min(5000, len(t)), random_state=0))
h.notna().mean().sort_values(ascending=False).round(3)

In [ ]:
# sample images (after download_images has run)
from PIL import Image
from src.download_images import image_path
img_dir = get_path(cfg, "images_dir") / "train"
rows = train.sample(8, random_state=1)
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax, (_, r) in zip(axes, rows.iterrows()):
    p = image_path(img_dir, r[cols["id"]], r[cols["image_url"]])
    if p.exists():
        ax.imshow(Image.open(p))
    ax.set_title(str(r.get(cols["target"], ""))[:12]); ax.axis("off")
plt.show()